# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [2]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [3]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['CASXNGCMTJ', 'RTMQGCGXND'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 3,  1, 19, 24, 14,  7,  3, 13, 20, 10],
       [18, 20, 13, 17,  7,  3,  7, 24, 14,  4]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 10, 20, 13,  3,  7, 14, 24, 19,  1],
       [ 0,  4, 14, 24,  7,  3,  7, 17, 13, 20]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[10, 20, 13,  3,  7, 14, 24, 19,  1,  3],
       [ 4, 14, 24,  7,  3,  7, 17, 13, 20, 18]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [ ]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        todo        
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        # 1. 嵌入
        enc_emb = self.embed_layer(enc_ids)  # (batch, enc_len, 64)
        dec_emb = self.embed_layer(dec_ids)  # (batch, dec_len, 64)        
        # 2. 编码
        enc_out, enc_state = self.encoder(enc_emb)  # enc_out: (batch, enc_len, 128)        
        # 3. 解码（用编码器的最后状态初始化）
        dec_out, dec_state = self.decoder(dec_emb, initial_state=enc_state)  # dec_out: (batch, dec_len, 128)        
        # 4. 双线性注意力: score = Q^T · W · K
        query = self.dense_attn(dec_out)  # (batch, dec_len, 128)      
        scores = tf.matmul(query, enc_out, transpose_b=True)  # 计算注意力分数：查询 × 键的转置
        alpha = tf.nn.softmax(scores, axis=-1)   # softmax 得到注意力权重
        attn = tf.matmul(alpha, enc_out)  # 加权求和得到上下文向量        
        # 5. 融合解码器输出和上下文向量
        fused = tf.concat([dec_out, attn], axis=-1)  # (batch, dec_len, 256)       
        # 6. 输出层
        logits = self.dense(fused)  # (batch, dec_len, 27)
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        单步解码，用于预测时
        shape(x) = [batch_size,]
        state: 解码器当前状态 (batch, 128)
        enc_out: 编码器所有输出 (batch, enc_len, 128)
        todo 参考sequence_reversal-exercise, 自己构建单步解码逻辑
        '''
        # 1. 嵌入当前输入
        inp_emb = self.embed_layer(x)  # (batch, 64)        
        # 2. 解码器一步
        dec_output, new_state = self.decoder_cell.call(inp_emb, state)  # dec_output: (batch, 128)        
        # 3. 双线性注意力
        query = tf.expand_dims(self.dense_attn(dec_output), axis=1)  # 添加时间步维度，匹配矩阵乘法要求       
        # 计算注意力分数
        scores = tf.matmul(query, enc_out, transpose_b=True)  # (batch, 1, enc_len)        
        # softmax 得到权重
        alpha = tf.nn.softmax(scores, axis=-1)  # (batch, 1, enc_len)        
        # 加权求和得到上下文向量
        attn = tf.matmul(alpha, enc_out)  # (batch, 1, 128)
        attn = tf.squeeze(attn, axis=1)   # (batch, 128)        
        # 4. 融合
        fused = tf.concat([dec_output, attn], axis=-1)  # (batch, 256)        
        # 5. 输出层，预测下一个token
        logits = self.dense(fused)  # (batch, 27)
        out = tf.argmax(logits, axis=-1)  # (batch,)    
        return out, new_state

# Loss函数以及训练逻辑

In [5]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [6]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3036551
step 500 : loss 1.376687
step 1000 : loss 0.25864604
step 1500 : loss 0.09831582


<tf.Tensor: shape=(), dtype=float32, numpy=0.037118606>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [7]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, False, True, False, True, True, True, True, True, True, False, True, True, False, True, True, True, True, True, True, True, True, True]
[('EDOXHJYIQCLAIONNWVFQ', 'QFVWNNOIALCQIYJHXODE'), ('ULELTHSODUMZPTDDSVKY', 'YKVSDDTPZMUDOSHTLELU'), ('VQXLJFXHGRTTBMEGAHYO', 'OYHAGEMBTTRGHXFJLXQV'), ('WJRZGJUHLAUDMYCKQEVD', 'DVEQKCYMDUALHUJGZRJW'), ('INJISVXADNSTMRYQCGKT', 'TKGCQYRMTSNDAXVSIJNI'), ('GDDMSAQMJPZFZGJSNPGN', 'NGPNSJGZFZPJMQASMDDC'), ('RCYZEVFBOPCMPMGFZHPA', 'APHZFGMPMCPOBFVEZYCR'), ('ZMQPNDVFQYCGJRKBLJWB', 'BWJLBKRJGCYQFVDNPQMZ'), ('EDFJAAYWBDRGUHIISNJP', 'PJNSIIHUGRDBWYAAJFDE'), ('LQBWJORZHKIAQNOGAIEA', 'AEIAGONQAIKHZROJWBQL'), ('AZZAAHIOVYTMIRKUZIHI', 'IHIZUKRIMTYVOIHAAZZA'), ('VYAGPKVOYEPNAVHHETMD', 'DMTEHHVANPEYOVKPGAYV'), ('KVBWJTVKVBWJNTBZOZVK', 'KVZOZBTNJWBVKVTVXZFS'), ('GWBIRGUWXIQJWFZTIKXN', 'NXKITZFWJQIXWUGRIBWG'), ('MJIHXMFJIHISYTWSSEJL', 'LJESSWTYSIHIJFMXHKRR'), ('SEOVBWOSCFAPYMFASASL', 'LSASAFMYPAFCSOWBVOEH'), ('